# sdiff demo

Diffs two structured OpenAPI files and classifies every change as **BREAKING**, **BEHAVIOUR**, or **COSMETIC**.
See [README.md](README.md) for how it works. This notebook runs it end to end against a real, bundled example.

You need one API key:
- `TYPESAFE_API_KEY` for the default Jev judge, or
- `OPENROUTER_API_KEY` (or `OPENAI_API_KEY`) for the cheap chat fallback (`--judge chat`)

Put either in a `.env` file next to this notebook (see `.env.example`), or paste it when prompted below.

In [ ]:
%pip install -q -e .

In [ ]:
import os
from getpass import getpass

import sdiff  # loads .env automatically if present

if not os.environ.get("TYPESAFE_API_KEY") and not (os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPENAI_API_KEY")):
    os.environ["OPENROUTER_API_KEY"] = getpass("No key found in .env - paste an OpenRouter or TypeSafe key: ")

## 1. Deterministic parse + align (no AI)

`sdiff.diff()` flattens both files and reports every leaf that changed, was added, or removed. This step is pure code, runs instantly, and needs no API key.

In [ ]:
old = sdiff.load("examples/stripe-v2440.yaml")
new = sdiff.load("examples/stripe-v2442.yaml")

changes = sdiff.diff(old, new)
print(f"{len(changes)} changed leaves\n")
for path, before, after in changes[:10]:
    print(f"  {path}: {before} -> {after}")

## 2. Classify with the chat judge (cheap, works with any OpenRouter/OpenAI-compatible key)

In [ ]:
judge = sdiff.ChatJudge()  # defaults to OpenRouter, openai/gpt-5-nano
results = sdiff.run(old, new, judge)
sdiff.render(results, json_output=False)

## 3. Same run with Jev (the real judge this tool is built around)

Needs `TYPESAFE_API_KEY`. Swapping the judge is the whole diff between this cell and the one above.

In [ ]:
if os.environ.get("TYPESAFE_API_KEY"):
    jev_results = sdiff.run(old, new, sdiff.JevJudge())
    sdiff.render(jev_results, json_output=False)
else:
    print("Set TYPESAFE_API_KEY (in .env or os.environ) to try the Jev judge.")

## 4. Try your own spec pair

Swap the paths below for any two OpenAPI JSON/YAML files, e.g. two versions of your own spec, or the CLI: `python sdiff.py old.yaml new.yaml --judge chat`.

In [ ]:
my_old = sdiff.load("examples/stripe-v2440.yaml")  # <- replace
my_new = sdiff.load("examples/stripe-v2442.yaml")  # <- replace

my_results = sdiff.run(my_old, my_new, sdiff.ChatJudge())
sdiff.render(my_results, json_output=False)
exit_code = 1 if any(r["kind"] == "breaking" for r in my_results) else 0
print(f"\nexit code: {exit_code}")